## RAG Pipeline:- 1) Data Ingestion to VectorDB

In [1]:
import os
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/qv/_yq6cw0d01bgq375mdy816400000gn/T/ipykernel_65953/582141364.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
/Users/kaushik1707/Desktop/Trust-RAG/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Read all pdf files in the directory

def process_all_pdfs(pdf_directory):
    '''ALL pdf files will be processed'''

    all_documents = []
    pdf_dir=Path(pdf_directory)

    ## Find all pdf files
    pdf_files = list(pdf_dir.glob('**/*.pdf'))
    print(f'Found {len(pdf_files)} PDF files to process')

    for pdf_file in pdf_files:
        print(f'\nProcessing: {pdf_file.name}')
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            ## Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f'Loaded {len(documents)} pages')

        except Exception as e:
            print(f'Error: {e}')

    print(f'\nTotal documents loaded; {len(all_documents)}')
    return all_documents

## Process all PDFs in data directory
all_pdf_documents = process_all_pdfs('../data')

    
    

Found 4 PDF files to process

Processing: Resume_Meghana.pdf
Loaded 1 pages

Processing: shyam resume.pdf
Loaded 3 pages

Processing: Kaushik_ML_Resume.pdf
Loaded 1 pages

Processing: Kaushik_ML_DEV_Resume.pdf
Loaded 1 pages

Total documents loaded; 6


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-13T07:48:19+00:00', 'source': '../data/pdf/Resume_Meghana.pdf', 'file_path': '../data/pdf/Resume_Meghana.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': 'Administrator', 'subject': '', 'keywords': '', 'moddate': '2026-05-13T07:48:19+00:00', 'trapped': '', 'modDate': 'D:20260513074819Z', 'creationDate': "D:20260513074819+00'00'", 'page': 0, 'source_file': 'Resume_Meghana.pdf', 'file_type': 'pdf'}, page_content='Meghana Perada                                                                                                 +91-8374955043 \nRoll No.:23011M2102                                                                                                       \nmeghanaperada9@gmail.com \nBachelor of Technology                                                                             linkedin.com/in/meghana-perada-035639348 \nJawaharlal Nehru Technological Un

### Chunking using text_splitter(RecursiveCharacterTextSplitter)

In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,  ##size of chunk
        chunk_overlap=chunk_overlap,    ##overlap 200 character from previous chunk into present chunk
        length_function=len,    ##use python len() function to count no.of characters
        separators=["\n\n", "\n", " ", ""]  ##seperate chunks based on para, lines, space, character in priority wise
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunks=split_documents(all_pdf_documents)
chunks

Split 6 documents into 18 chunks

Example chunk:
Content: Meghana Perada                                                                                                 +91-8374955043 
Roll No.:23011M2102                                                      ...
Metadata: {'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-13T07:48:19+00:00', 'source': '../data/pdf/Resume_Meghana.pdf', 'file_path': '../data/pdf/Resume_Meghana.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': 'Administrator', 'subject': '', 'keywords': '', 'moddate': '2026-05-13T07:48:19+00:00', 'trapped': '', 'modDate': 'D:20260513074819Z', 'creationDate': "D:20260513074819+00'00'", 'page': 0, 'source_file': 'Resume_Meghana.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-13T07:48:19+00:00', 'source': '../data/pdf/Resume_Meghana.pdf', 'file_path': '../data/pdf/Resume_Meghana.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': 'Administrator', 'subject': '', 'keywords': '', 'moddate': '2026-05-13T07:48:19+00:00', 'trapped': '', 'modDate': 'D:20260513074819Z', 'creationDate': "D:20260513074819+00'00'", 'page': 0, 'source_file': 'Resume_Meghana.pdf', 'file_type': 'pdf'}, page_content='Meghana Perada                                                                                                 +91-8374955043 \nRoll No.:23011M2102                                                                                                       \nmeghanaperada9@gmail.com \nBachelor of Technology                                                                             linkedin.com/in/meghana-perada-035639348 \nJawaharlal Nehru Technological Un

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None ## store the loaded SentenceTransformer model inside the EmbeddingManager object.
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


## initialize the embedding manager

embedding_manager=EmbeddingManager()
embedding_manager


Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9645.95it/s]


Model loaded successfully. Embedding dimension: 384


/var/folders/qv/_yq6cw0d01bgq375mdy816400000gn/T/ipykernel_65953/2368076259.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


## VectorStore

In [8]:
import os

In [9]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [10]:
chunks

[Document(metadata={'producer': 'www.ilovepdf.com', 'creator': 'Microsoft® Word 2016', 'creationdate': '2026-05-13T07:48:19+00:00', 'source': '../data/pdf/Resume_Meghana.pdf', 'file_path': '../data/pdf/Resume_Meghana.pdf', 'total_pages': 1, 'format': 'PDF 1.5', 'title': '', 'author': 'Administrator', 'subject': '', 'keywords': '', 'moddate': '2026-05-13T07:48:19+00:00', 'trapped': '', 'modDate': 'D:20260513074819Z', 'creationDate': "D:20260513074819+00'00'", 'page': 0, 'source_file': 'Resume_Meghana.pdf', 'file_type': 'pdf'}, page_content='Meghana Perada                                                                                                 +91-8374955043 \nRoll No.:23011M2102                                                                                                       \nmeghanaperada9@gmail.com \nBachelor of Technology                                                                             linkedin.com/in/meghana-perada-035639348 \nJawaharlal Nehru Technological Un

In [11]:
##converting chuks into embeddings
texts = [doc.page_content for doc in chunks]

##sending texts for embedding
embeddings = embedding_manager.generate_embeddings(texts)

##stroing actual data with their corresponding embeddings into VectorDB
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 18 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.40it/s]

Generated embeddings with shape: (18, 384)
Adding 18 documents to vector store...
Successfully added 18 documents to vector store
Total documents in collection: 18


In [14]:
data = vectorstore.collection.get()

print("Number of documents:", len(data["ids"]))

for i in range(len(data["ids"])):
    print(f"\n========== {i+1} ==========")
    print("ID:", data["ids"][i])
    print("Document:", data["documents"][i])
    print("Metadata:", data["metadatas"][i])

Number of documents: 18

========== 1 ==========
ID: doc_d65b2788_0
Document: Meghana Perada                                                                                                 +91-8374955043 
Roll No.:23011M2102                                                                                                       
meghanaperada9@gmail.com 
Bachelor of Technology                                                                             linkedin.com/in/meghana-perada-035639348 
Jawaharlal Nehru Technological University, Hyderabad 
 
EDUCATIOn 
 
• Bachelor of Technology in Computer Science and Engineering 
2023-27 
Jawaharlal  Nehru  Technological  University , Hyderabd 
CGPA: 8.32 
PERsonaL PROjECTs 
 
• RAG Document Q&A System 
Developed a high performance RAG system for intelligent document question-answering using hybrid semantic and keyword-based 
retrieval 
– Implemented FAISS and BM25-based hybrid retrieval to improve contextual relevance and answer accuracy.
Metadat